# 🌱 Green AI in Practice: Data-Centric Techniques for Energy-Efficient Training

**Task:** Spam detection | **Technique:** Dataset size reduction | **Metric:** Energy (J) vs. Performance (F1)

Training costs energy. One of the simplest ways to reduce that cost is to train on less data. 
In this notebook you will train the same spam detection model on four different dataset sizes 
and measure the energy consumed each time. Your goal is to explore where the energy saving is 
real and the performance loss is acceptable.

**You will measure:** energy (kWh), training time (s), accuracy, F1 score

> ⚠️ **Run cells in order.** Do not interrupt a running cell, the energy tracker needs 
> the full run to complete.


## 1 · Install Libraries

In [8]:
from joblib import load
from text_preprocessing import _load_data
from codecarbon import EmissionsTracker
import csv
import time
import random

from text_classification import generate_model, model_validation
from modify_dataset import modify_dataset_and_raw_data_with_percentage_size_to_keep
from modify_dataset import modify_dataset_select_features

from sklearn.tree import DecisionTreeClassifier
from energy_utils import energy_stats

import logging
logging.getLogger("codecarbon").setLevel(logging.ERROR)

import numpy as np

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

## 2 · Load Dataset

Loads the **SMS Spam Collection** (5,574 messages, ~87% ham / ~13% spam).

The test set is fixed here and will **not change** across runs — this is essential for 
fair comparison between training sizes.

> 📊 **Note:** The dataset is imbalanced. Use F1 score, not accuracy alone, 
> to evaluate model performance.

In [9]:
raw_data = _load_data()
preprocessed_data = load('output/preprocessed_data.joblib')

## 3 · Baseline: Full Dataset (100%)

Trains a Decision Tree classifier using the full training set.
CodeCarbon measures energy between `tracker.start()` and `tracker.stop()`.


In [ ]:
classifier = DecisionTreeClassifier()

training_tracker = EmissionsTracker(save_to_file=False, allow_multiple_runs=True, log_level="error")
#### START TIMED TRAINING SECTION ####
training_tracker.start()
classifier, X_train, X_test, y_train, y_test, test_messages = generate_model(classifier, raw_data,
                                                                                 preprocessed_data)
training_energy_consumption_kwh = training_tracker.stop()
#### STOP TIMED TRAINING SECTION ####
training_energy_consumption, training_duration = energy_stats(training_energy_consumption_kwh, training_tracker)
_, scores, report = model_validation(classifier, X_test, y_test)

print(f"  Energy Consumption: {training_energy_consumption:.4f} Joules")
print(f"  Duration: {training_duration:.2f} seconds")
print(f"  F1 Score: {scores['f1']:.2f}")
print(f"  Accuracy: {scores['accuracy']:.2f}")


## 4 · Reduced Dataset Runs

Same model, same random seed. Explore with smaller training sets.

Modify the value of ```PERCENTAGE_DATASET``` to reduce the dataset size (percentage of the full dataset to use).


In [ ]:
PERCENTAGE_DATASET = 80

modified_preprocessed_data, modified_raw_data = modify_dataset_and_raw_data_with_percentage_size_to_keep(
        preprocessed_data, raw_data, PERCENTAGE_DATASET)

training_tracker = EmissionsTracker(save_to_file=False, allow_multiple_runs=True, log_level="error")
#### START TIMED TRAINING SECTION ####
training_tracker.start()
classifier, X_train, X_test, y_train, y_test, test_messages = generate_model(classifier, modified_raw_data,
                                                                                 modified_preprocessed_data)
training_energy_consumption_kwh = training_tracker.stop()
#### STOP TIMED TRAINING SECTION ####
training_energy_consumption, training_duration = energy_stats(training_energy_consumption_kwh, training_tracker)
_, scores, report = model_validation(classifier, X_test, y_test)

print(f"  Energy Consumption: {training_energy_consumption:.4f} Joules")
print(f"  Duration: {training_duration:.2f} seconds")
print(f"  F1 Score: {scores['f1']:.2f}")
print(f"  Accuracy: {scores['accuracy']:.2f}")